# Load data

In [27]:
import numpy as np
import pandas as pd
from pathlib import Path

In [28]:
data_path = Path('../data/interim/monthly_panel.csv')
df = pd.read_csv(data_path)
df['month'] = pd.to_datetime(df['month']).dt.to_period('M')

df

,date,ticker,open,high,low,close,adj_close,volume,dividends,stock_splits,month,return,target_return
0,2010-01-29,A,20.693848,20.865522,19.971388,20.050072,17.707222,5923046.0,0.0,0.0,2010-01,NaN,0.122369
1,2010-02-26,A,22.296137,22.546495,22.160229,22.503576,19.874033,4678127.0,0.0,0.0,2010-02,0.122369,0.093134
2,2010-03-31,A,24.713877,24.735336,24.527897,24.599428,21.724976,5293387.0,0.0,0.0,2010-03,0.093134,0.054376
3,2010-04-30,A,26.645206,26.773964,25.915594,25.937054,22.906301,4195678.0,0.0,0.0,2010-04,0.054376,-0.107557
4,2010-05-28,A,23.433475,23.497854,23.004292,23.147352,20.442572,3430692.0,0.0,0.0,2010-05,-0.107557,-0.121446
...,...,...,...,...,...,...,...,...,...,...,...,...,...
94220,2026-04-30,ZTS,114.040001,115.540001,113.790001,114.970001,114.174103,4297200.0,0.0,0.0,2026-04,-0.023178,-0.324258
94221,2026-05-29,ZTS,78.510002,78.879997,77.440002,77.690002,77.152184,9295100.0,0.0,0.0,2026-05,-0.324258,-0.075042
94222,2026-06-30,ZTS,73.169998,73.559998,71.470001,71.860001,71.362541,8322200.0,0.0,0.0,2026-06,-0.075042,0.083061
94223,2026-07-31,ZTS,75.610001,77.519997,74.910004,77.290001,77.290001,5018300.0,0.0,0.0,2026-07,0.083061,-0.002458


In [29]:
df.describe()

,open,high,low,close,adj_close,volume,dividends,stock_splits,return,target_return
count,94225.000000,94225.000000,94225.000000,94225.000000,94225.000000,9.422500e+04,94225.000000,94225.000000,93722.000000,93722.000000
mean,111.900996,113.178331,110.636361,111.936314,102.677109,7.914932e+06,0.011184,0.000266,0.015192,0.015192
std,235.639085,238.384420,233.161596,235.909161,235.360987,3.085204e+07,0.134902,0.028693,0.088768,0.088768
min,0.226500,0.233750,0.223000,0.229750,0.210151,0.000000e+00,0.000000,0.000000,-0.832263,-0.832263
25%,35.220001,35.619999,34.849998,35.209999,28.112082,1.203100e+06,0.000000,0.000000,-0.033848,-0.033848
50%,65.660004,66.370003,64.940002,65.669998,54.775562,2.650300e+06,0.000000,0.000000,0.013930,0.013930
75%,122.959999,124.330002,121.577927,123.019997,110.524567,6.153800e+06,0.000000,0.000000,0.060288,0.060288
max,9729.759766,9844.730469,9672.500000,9811.799805,9811.799805,1.245952e+09,22.000000,5.000000,2.138440,2.138440


# Price

In [30]:
# Momentum
def compounded_return(x: np.ndarray):
    return np.prod(1+x)-1

g_mom = df.groupby('ticker')['return']

df['mom_1m'] = g_mom.rolling(1).apply(compounded_return, raw=False).reset_index(drop=True)
df['mom_3m'] = g_mom.rolling(3).apply(compounded_return, raw=False).reset_index(drop=True)
df['mom_6m'] = g_mom.rolling(6).apply(compounded_return, raw=False).reset_index(drop=True)
df['mom_12m'] = g_mom.rolling(12).apply(compounded_return, raw=False).reset_index(drop=True)
df['mom_12m_1'] = g_mom.shift(1).rolling(11).apply(compounded_return, raw=False).reset_index(drop=True)

# Reversal
df['reversal_1m'] = -df['return']

df.head()

,date,ticker,open,high,low,close,adj_close,volume,dividends,stock_splits,month,return,target_return,mom_1m,mom_3m,mom_6m,mom_12m,mom_12m_1,reversal_1m
0,2010-01-29,A,20.693848,20.865522,19.971388,20.050072,17.707222,5923046.0,0.0,0.0,2010-01,NaN,0.122369,NaN,NaN,NaN,NaN,NaN,NaN
1,2010-02-26,A,22.296137,22.546495,22.160229,22.503576,19.874033,4678127.0,0.0,0.0,2010-02,0.122369,0.093134,0.122369,NaN,NaN,NaN,NaN,-0.122369
2,2010-03-31,A,24.713877,24.735336,24.527897,24.599428,21.724976,5293387.0,0.0,0.0,2010-03,0.093134,0.054376,0.093134,NaN,NaN,NaN,NaN,-0.093134
3,2010-04-30,A,26.645206,26.773964,25.915594,25.937054,22.906301,4195678.0,0.0,0.0,2010-04,0.054376,-0.107557,0.054376,0.293614,NaN,NaN,NaN,-0.054376
4,2010-05-28,A,23.433475,23.497854,23.004292,23.147352,20.442572,3430692.0,0.0,0.0,2010-05,-0.107557,-0.121446,-0.107557,0.028607,NaN,NaN,NaN,0.107557


In [31]:
# Import daily data
df_daily_path = Path('../data/raw/sp500_2010_2026.csv')
df_daily = pd.read_csv(df_daily_path)

df_daily = df_daily.sort_values(by=['ticker','date'])
df_daily['month'] = pd.to_datetime(df_daily['date']).dt.to_period('M')

df_daily.head()

,date,ticker,open,high,low,close,adj_close,volume,dividends,stock_splits,month
0,2010-01-04,A,22.453505,22.625179,22.267525,22.389128,19.772945,3815561.0,0.0,0.0,2010-01
1,2010-01-05,A,22.324751,22.331903,22.002861,22.145924,19.558168,4186031.0,0.0,0.0,2010-01
2,2010-01-06,A,22.067240,22.174536,22.002861,22.067240,19.488678,3243779.0,0.0,0.0,2010-01
3,2010-01-07,A,22.017166,22.045780,21.816881,22.038628,19.463406,3095172.0,0.0,0.0,2010-01
4,2010-01-08,A,21.917025,22.067240,21.745352,22.031473,19.457100,3733918.0,0.0,0.0,2010-01


In [32]:
g_ma = df_daily.groupby('ticker')['adj_close']

df_daily['ma_20d'] = g_ma.rolling(20).mean().reset_index(drop=True)
df_daily['ma_60d'] = g_ma.rolling(60).mean().reset_index(drop=True)
df_daily['ma_200d'] = g_ma.rolling(200).mean().reset_index(drop=True)

df_daily['ma_dist_20d'] = df_daily['adj_close'] / df_daily['ma_20d'] - 1
df_daily['ma_dist_60d'] = df_daily['adj_close'] / df_daily['ma_60d'] - 1
df_daily['ma_dist_200d'] = df_daily['adj_close'] / df_daily['ma_200d'] - 1

df_ma_dist = df_daily.groupby(['ticker', 'month'])[['ticker','month','ma_dist_20d', 'ma_dist_60d', 'ma_dist_200d']].tail(1).reset_index(drop=True)

df = df.merge(df_ma_dist, how='left', on=['ticker','month'])

df.head()

,date,ticker,open,high,low,close,adj_close,volume,dividends,stock_splits,...,target_return,mom_1m,mom_3m,mom_6m,mom_12m,mom_12m_1,reversal_1m,ma_dist_20d,ma_dist_60d,ma_dist_200d
0,2010-01-29,A,20.693848,20.865522,19.971388,20.050072,17.707222,5923046.0,0.0,0.0,...,0.122369,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2010-02-26,A,22.296137,22.546495,22.160229,22.503576,19.874033,4678127.0,0.0,0.0,...,0.093134,0.122369,NaN,NaN,NaN,NaN,-0.122369,0.047829,NaN,NaN
2,2010-03-31,A,24.713877,24.735336,24.527897,24.599428,21.724976,5293387.0,0.0,0.0,...,0.054376,0.093134,NaN,NaN,NaN,NaN,-0.093134,0.021475,0.094415,NaN
3,2010-04-30,A,26.645206,26.773964,25.915594,25.937054,22.906301,4195678.0,0.0,0.0,...,-0.107557,0.054376,0.293614,NaN,NaN,NaN,-0.054376,0.022647,0.088780,NaN
4,2010-05-28,A,23.433475,23.497854,23.004292,23.147352,20.442572,3430692.0,0.0,0.0,...,-0.121446,-0.107557,0.028607,NaN,NaN,NaN,0.107557,-0.025624,-0.052412,NaN


In [33]:
# Realized volatility
df_daily['return'] = df_daily.groupby('ticker')['adj_close'].pct_change(1)

g_rvol = df_daily.groupby('ticker')['return']

df_daily['rvol_21d'] = g_rvol.rolling(21).std(ddof=1).reset_index(drop=True) * np.sqrt(252)
df_daily['rvol_63d'] = g_rvol.rolling(63).std(ddof=1).reset_index(drop=True) * np.sqrt(252)
df_daily['rvol_252d'] = g_rvol.rolling(252).std(ddof=1).reset_index(drop=True) * np.sqrt(252)

df_rvol = df_daily.groupby(['ticker','month'])[['ticker','month','rvol_21d','rvol_63d','rvol_252d']].tail(1)

df = df.merge(df_rvol, how='left', on=['ticker','month'])

df.head()

,date,ticker,open,high,low,close,adj_close,volume,dividends,stock_splits,...,mom_6m,mom_12m,mom_12m_1,reversal_1m,ma_dist_20d,ma_dist_60d,ma_dist_200d,rvol_21d,rvol_63d,rvol_252d
0,2010-01-29,A,20.693848,20.865522,19.971388,20.050072,17.707222,5923046.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2010-02-26,A,22.296137,22.546495,22.160229,22.503576,19.874033,4678127.0,0.0,0.0,...,NaN,NaN,NaN,-0.122369,0.047829,NaN,NaN,0.220318,NaN,NaN
2,2010-03-31,A,24.713877,24.735336,24.527897,24.599428,21.724976,5293387.0,0.0,0.0,...,NaN,NaN,NaN,-0.093134,0.021475,0.094415,NaN,0.145493,NaN,NaN
3,2010-04-30,A,26.645206,26.773964,25.915594,25.937054,22.906301,4195678.0,0.0,0.0,...,NaN,NaN,NaN,-0.054376,0.022647,0.088780,NaN,0.234837,0.192332,NaN
4,2010-05-28,A,23.433475,23.497854,23.004292,23.147352,20.442572,3430692.0,0.0,0.0,...,NaN,NaN,NaN,0.107557,-0.025624,-0.052412,NaN,0.506016,0.330592,NaN


In [34]:
# Downside volatility
df_daily['downside_return'] = df_daily['return'].clip(upper=0)

g_dvol = df_daily.groupby('ticker')['downside_return']

df_daily['dvol_21d'] = g_dvol.rolling(21).std(ddof=1).reset_index(drop=True) * np.sqrt(252)
df_daily['dvol_63d'] = g_dvol.rolling(63).std(ddof=1).reset_index(drop=True) * np.sqrt(252)
df_daily['dvol_252d'] = g_dvol.rolling(252).std(ddof=1).reset_index(drop=True) * np.sqrt(252)

df_dvol = df_daily.groupby(['ticker','month'])[['ticker','month','dvol_21d','dvol_63d','dvol_252d']].tail(1)

df = df.merge(df_dvol, how='left', on=['ticker','month'])

df.head()

,date,ticker,open,high,low,close,adj_close,volume,dividends,stock_splits,...,reversal_1m,ma_dist_20d,ma_dist_60d,ma_dist_200d,rvol_21d,rvol_63d,rvol_252d,dvol_21d,dvol_63d,dvol_252d
0,2010-01-29,A,20.693848,20.865522,19.971388,20.050072,17.707222,5923046.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2010-02-26,A,22.296137,22.546495,22.160229,22.503576,19.874033,4678127.0,0.0,0.0,...,-0.122369,0.047829,NaN,NaN,0.220318,NaN,NaN,0.107623,NaN,NaN
2,2010-03-31,A,24.713877,24.735336,24.527897,24.599428,21.724976,5293387.0,0.0,0.0,...,-0.093134,0.021475,0.094415,NaN,0.145493,NaN,NaN,0.041789,NaN,NaN
3,2010-04-30,A,26.645206,26.773964,25.915594,25.937054,22.906301,4195678.0,0.0,0.0,...,-0.054376,0.022647,0.088780,NaN,0.234837,0.192332,NaN,0.108467,0.07801,NaN
4,2010-05-28,A,23.433475,23.497854,23.004292,23.147352,20.442572,3430692.0,0.0,0.0,...,0.107557,-0.025624,-0.052412,NaN,0.506016,0.330592,NaN,0.318332,0.21085,NaN


In [35]:
# Max drawdown
def mdd(x):
    return np.min(x / np.maximum.accumulate(x) - 1)

g_mdd = df_daily.groupby('ticker')['adj_close']

df_daily['mdd_63d'] = g_mdd.rolling(63).apply(mdd, raw=True).reset_index(drop=True)
df_daily['mdd_252d'] = g_mdd.rolling(252).apply(mdd, raw=True).reset_index(drop=True)

df_mdd = df_daily.groupby(['ticker','month'])[['ticker','month','mdd_63d','mdd_252d']].tail(1)

df = df.merge(df_mdd, on=['ticker','month'], how='left')

df.head()

,date,ticker,open,high,low,close,adj_close,volume,dividends,stock_splits,...,ma_dist_60d,ma_dist_200d,rvol_21d,rvol_63d,rvol_252d,dvol_21d,dvol_63d,dvol_252d,mdd_63d,mdd_252d
0,2010-01-29,A,20.693848,20.865522,19.971388,20.050072,17.707222,5923046.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2010-02-26,A,22.296137,22.546495,22.160229,22.503576,19.874033,4678127.0,0.0,0.0,...,NaN,NaN,0.220318,NaN,NaN,0.107623,NaN,NaN,NaN,NaN
2,2010-03-31,A,24.713877,24.735336,24.527897,24.599428,21.724976,5293387.0,0.0,0.0,...,0.094415,NaN,0.145493,NaN,NaN,0.041789,NaN,NaN,NaN,NaN
3,2010-04-30,A,26.645206,26.773964,25.915594,25.937054,22.906301,4195678.0,0.0,0.0,...,0.088780,NaN,0.234837,0.192332,NaN,0.108467,0.07801,NaN,-0.026421,NaN
4,2010-05-28,A,23.433475,23.497854,23.004292,23.147352,20.442572,3430692.0,0.0,0.0,...,-0.052412,NaN,0.506016,0.330592,NaN,0.318332,0.21085,NaN,-0.166264,NaN


In [36]:
# Max/min return
g_max_min_ret = df_daily.groupby('ticker')['return']

df_daily['max_ret_21d'] = g_max_min_ret.rolling(21).max().reset_index(drop=True)
df_daily['min_ret_21d'] = g_max_min_ret.rolling(21).min().reset_index(drop=True)

df_max_min_ret = df_daily.groupby(['ticker','month'])[['ticker','month','max_ret_21d','min_ret_21d']].tail(1)

df = df.merge(df_max_min_ret, on=['ticker','month'], how='left')

df.head()

,date,ticker,open,high,low,close,adj_close,volume,dividends,stock_splits,...,rvol_21d,rvol_63d,rvol_252d,dvol_21d,dvol_63d,dvol_252d,mdd_63d,mdd_252d,max_ret_21d,min_ret_21d
0,2010-01-29,A,20.693848,20.865522,19.971388,20.050072,17.707222,5923046.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2010-02-26,A,22.296137,22.546495,22.160229,22.503576,19.874033,4678127.0,0.0,0.0,...,0.220318,NaN,NaN,0.107623,NaN,NaN,NaN,NaN,0.039243,-0.023004
2,2010-03-31,A,24.713877,24.735336,24.527897,24.599428,21.724976,5293387.0,0.0,0.0,...,0.145493,NaN,NaN,0.041789,NaN,NaN,NaN,NaN,0.027899,-0.006947
3,2010-04-30,A,26.645206,26.773964,25.915594,25.937054,22.906301,4195678.0,0.0,0.0,...,0.234837,0.192332,NaN,0.108467,0.07801,NaN,-0.026421,NaN,0.035455,-0.026054
4,2010-05-28,A,23.433475,23.497854,23.004292,23.147352,20.442572,3430692.0,0.0,0.0,...,0.506016,0.330592,NaN,0.318332,0.21085,NaN,-0.166264,NaN,0.072796,-0.062006


# Liquidity

In [37]:
# Average dollar volume
df_daily['dollar_volume'] = df_daily['close'] * df_daily['volume']

df_daily['avg_dollar_volume_21d'] = df_daily.groupby('ticker')['dollar_volume'].rolling(21).mean().reset_index(drop=True)
# Take log as dollar volume may be right-skewed
df_daily['log_avg_dollar_volume_21d'] = np.log(df_daily['avg_dollar_volume_21d'])

df_avg_daily_volume = df_daily.groupby(['ticker','month'])[['ticker','month','avg_dollar_volume_21d','log_avg_dollar_volume_21d']].tail(1)

df = df.merge(df_avg_daily_volume, on=['ticker','month'], how='left')

df.head()

/home/victor/Projects/ml-cross-sectional-alpha-research/.env/lib/python3.13/site-packages/pandas/core/arraylike.py:402: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


,date,ticker,open,high,low,close,adj_close,volume,dividends,stock_splits,...,rvol_252d,dvol_21d,dvol_63d,dvol_252d,mdd_63d,mdd_252d,max_ret_21d,min_ret_21d,avg_dollar_volume_21d,log_avg_dollar_volume_21d
0,2010-01-29,A,20.693848,20.865522,19.971388,20.050072,17.707222,5923046.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2010-02-26,A,22.296137,22.546495,22.160229,22.503576,19.874033,4678127.0,0.0,0.0,...,NaN,0.107623,NaN,NaN,NaN,NaN,0.039243,-0.023004,1.210369e+08,18.611606
2,2010-03-31,A,24.713877,24.735336,24.527897,24.599428,21.724976,5293387.0,0.0,0.0,...,NaN,0.041789,NaN,NaN,NaN,NaN,0.027899,-0.006947,1.251708e+08,18.645189
3,2010-04-30,A,26.645206,26.773964,25.915594,25.937054,22.906301,4195678.0,0.0,0.0,...,NaN,0.108467,0.07801,NaN,-0.026421,NaN,0.035455,-0.026054,1.281770e+08,18.668923
4,2010-05-28,A,23.433475,23.497854,23.004292,23.147352,20.442572,3430692.0,0.0,0.0,...,NaN,0.318332,0.21085,NaN,-0.166264,NaN,0.072796,-0.062006,1.448601e+08,18.791279


In [ ]:
# Volume shock_(21,63) = overline(V_21) / overline(V_63) - 1

g_volume_shock = df_daily.groupby('ticker')['volume']

df_daily['avg_volume_21d'] = g_volume_shock.rolling(21).mean().reset_index(drop=True)
df_daily['avg_volume_63d'] = g_volume_shock.rolling(63).mean().reset_index(drop=True)
df_daily['volume_shock'] = df_daily['avg_volume_21d'] / df_daily['avg_volume_63d'] - 1

df_volume_shock = df_daily.groupby(['ticker','month'])[['ticker','month','volume_shock']].tail(1)

df = df.merge(df_volume_shock, on=['ticker','month'], how='left')

df.head()

,date,ticker,open,high,low,close,adj_close,volume,dividends,stock_splits,...,dvol_21d,dvol_63d,dvol_252d,mdd_63d,mdd_252d,max_ret_21d,min_ret_21d,avg_dollar_volume_21d,log_avg_dollar_volume_21d,volume_shock
0,2010-01-29,A,20.693848,20.865522,19.971388,20.050072,17.707222,5923046.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2010-02-26,A,22.296137,22.546495,22.160229,22.503576,19.874033,4678127.0,0.0,0.0,...,0.107623,NaN,NaN,NaN,NaN,0.039243,-0.023004,1.210369e+08,18.611606,NaN
2,2010-03-31,A,24.713877,24.735336,24.527897,24.599428,21.724976,5293387.0,0.0,0.0,...,0.041789,NaN,NaN,NaN,NaN,0.027899,-0.006947,1.251708e+08,18.645189,NaN
3,2010-04-30,A,26.645206,26.773964,25.915594,25.937054,22.906301,4195678.0,0.0,0.0,...,0.108467,0.07801,NaN,-0.026421,NaN,0.035455,-0.026054,1.281770e+08,18.668923,-0.047372
4,2010-05-28,A,23.433475,23.497854,23.004292,23.147352,20.442572,3430692.0,0.0,0.0,...,0.318332,0.21085,NaN,-0.166264,NaN,0.072796,-0.062006,1.448601e+08,18.791279,0.116278


In [42]:
# Zero-return frequency
df_daily['zero_return'] = (df_daily['return']==0).astype(int)
df_daily['zero_ret_freq_21d'] = df_daily.groupby('ticker')['zero_return'].rolling(21).mean().reset_index(drop=True)

df_zero_ret_freq = df_daily.groupby(['ticker','month'])[['ticker','month','zero_ret_freq_21d']].tail(1)

df = df.merge(df_zero_ret_freq, on=['ticker','month'], how='left')

df.head()

,date,ticker,open,high,low,close,adj_close,volume,dividends,stock_splits,...,dvol_63d,dvol_252d,mdd_63d,mdd_252d,max_ret_21d,min_ret_21d,avg_dollar_volume_21d,log_avg_dollar_volume_21d,volume_shock,zero_ret_freq_21d
0,2010-01-29,A,20.693848,20.865522,19.971388,20.050072,17.707222,5923046.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2010-02-26,A,22.296137,22.546495,22.160229,22.503576,19.874033,4678127.0,0.0,0.0,...,NaN,NaN,NaN,NaN,0.039243,-0.023004,1.210369e+08,18.611606,NaN,0.0
2,2010-03-31,A,24.713877,24.735336,24.527897,24.599428,21.724976,5293387.0,0.0,0.0,...,NaN,NaN,NaN,NaN,0.027899,-0.006947,1.251708e+08,18.645189,NaN,0.0
3,2010-04-30,A,26.645206,26.773964,25.915594,25.937054,22.906301,4195678.0,0.0,0.0,...,0.07801,NaN,-0.026421,NaN,0.035455,-0.026054,1.281770e+08,18.668923,-0.047372,0.0
4,2010-05-28,A,23.433475,23.497854,23.004292,23.147352,20.442572,3430692.0,0.0,0.0,...,0.21085,NaN,-0.166264,NaN,0.072796,-0.062006,1.448601e+08,18.791279,0.116278,0.0
